# 🥇 Arquitetura Medalhão: Camada Gold (Business & Consumption Data)

**Projeto:** Foodhunter (RAG System para Recomendação de Restaurantes)
**Objetivo desta camada:** A camada Gold entrega o dado final, enriquecido e formatado especificamente para o consumo da aplicação (no nosso caso, o Banco de Dados Vetorial e o LLM).

**Neste Notebook, vamos:**
1. **Aplicar Regras de Negócio:** Criar um `gold_score` para ranquear os restaurantes de forma mais justa (balanceando a nota com a quantidade de avaliações).
2. **Engenharia de Prompt (RAG):** Criar a coluna `rag_context`, que é a frase exata em linguagem natural que será enviada ao LLM como contexto.
3. **Seleção de Features:** Manter apenas as colunas estritamente necessárias para a busca vetorial e exibição no front-end.

In [1]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler

# Configurações visuais
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

## 1. Ingestão da Camada Silver
Vamos carregar o arquivo `.parquet` limpo e estruturado que foi gerado na etapa anterior. Usamos caminhos relativos para manter o notebook funcional em qualquer máquina.

In [2]:
diretorio_atual = os.getcwd()
diretorio_raiz = os.path.dirname(diretorio_atual)

# Monta o caminho apontando para a pasta silver
caminho_silver = os.path.join(diretorio_raiz, 'silver', 'silver_data.parquet')

print(f"Lendo dados consolidados de: {caminho_silver}")
df = pd.read_parquet(caminho_silver)

print(f"Total de restaurantes processados: {len(df)}")
display(df.head(2))

Lendo dados consolidados de: c:\Faculdade - IA\Projeto AC2\RAG-Sapato\silver\silver_data.parquet
Total de restaurantes processados: 34987


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,categories,price,description,embedding,has_delivery,has_outdoor,price_range,embedding_list
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,Restaurants Food Bubble Tea Coffee & Tea Bakeries,1,Restaurants Food Bubble Tea Coffee & Tea Bakeries price:1,"[0.0035318147856742144, -0.008946051821112633, 0.04717320203781128, 0.03231402859091759, -0.06327924132347107, 0.0501706637442112, 0.0719829872250...",False,False,1,"[0.0035318147856742144, -0.008946051821112633, 0.04717320203781128, 0.03231402859091759, -0.06327924132347107, 0.0501706637442112, 0.0719829872250..."
1,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,Burgers Fast Food Sandwiches Food Ice Cream & Frozen Yogurt Restaurants,1,Burgers Fast Food Sandwiches Food Ice Cream & Frozen Yogurt Restaurants price:1,"[-0.005798297934234142, 0.013337074778974056, 0.05501795932650566, 0.03697891905903816, -0.007719673216342926, 0.041596420109272, -0.0066671497188...",True,True,1,"[-0.005798297934234142, 0.013337074778974056, 0.05501795932650566, 0.03697891905903816, -0.007719673216342926, 0.041596420109272, -0.0066671497188..."


## 2. Regra de Negócio: Score de Confiança (`gold_score`)
No mundo real, um restaurante com 5.0 estrelas e apenas 2 avaliações não é necessariamente melhor que um restaurante com 4.8 estrelas e 2.000 avaliações. 

Para resolver isso e ajudar o nosso RAG a recomendar os locais mais confiáveis, vamos criar um `gold_score`.
* Usamos o Logaritmo (`np.log1p`) nas avaliações para que restaurantes com 10.000 reviews não esmaguem os que têm 100.
* Normalizamos ambas as escalas (0 a 1).
* Damos peso de 70% para a Nota e 30% para a Popularidade.

In [3]:
# Suaviza a contagem de reviews
df['log_reviews'] = np.log1p(df['review_count'])

# Instancia o normalizador do scikit-learn
scaler = MinMaxScaler()

# Aplica a normalização nas colunas alvo
df[['stars_norm', 'reviews_norm']] = scaler.fit_transform(df[['stars', 'log_reviews']])

# Calcula o Score Final (0 a 1)
df['gold_score'] = (df['stars_norm'] * 0.7) + (df['reviews_norm'] * 0.3)

# Checagem visual do Top 5 restaurantes pelo nosso novo Score
df_top = df.sort_values(by='gold_score', ascending=False)
display(df_top[['name', 'stars', 'review_count', 'gold_score']].head(5))

,name,stars,review_count,gold_score
2824,Blues City Deli,5.0,991,0.914619
31217,Carlillos Cocina,5.0,799,0.905580
21327,Hattie B’s Hot Chicken - Nashville,4.5,6093,0.903393
33302,Reading Terminal Market,4.5,5721,0.900746
3820,Tumerico,5.0,705,0.900328


## 3. Preparação para o RAG (`rag_context`)
Modelos de linguagem (LLMs) não leem tabelas muito bem, eles leem **textos**. Para que o LLM do Foodhunter consiga justificar a recomendação para o usuário de forma natural, vamos concatenar os metadados em uma única string descritiva.

Essa coluna será o *Contexto* injetado no prompt.

In [4]:
# Criação do texto rico para o LLM
df['rag_context'] = df.apply(
    lambda row: f"O restaurante '{row['name']}' fica na cidade de {row['city']}. "
                f"Sua categoria é: {row['categories']}. "
                f"Avaliação do público: {row['stars']} estrelas baseadas em {row['review_count']} reviews. "
                f"Possui delivery? {'Sim' if row['has_delivery'] else 'Não'}. "
                f"Grau de recomendação interno do Foodhunter: {row['gold_score']:.2f}.", 
    axis=1
)

# Exibindo um exemplo de como o LLM vai ler esse dado
print("EXEMPLO DE CONTEXTO GERADO PARA O RAG:\n")
print(df['rag_context'].iloc[0])

EXEMPLO DE CONTEXTO GERADO PARA O RAG:

O restaurante 'St Honore Pastries' fica na cidade de Philadelphia. Sua categoria é: Restaurants  Food  Bubble Tea  Coffee & Tea  Bakeries. Avaliação do público: 4.0 estrelas baseadas em 80 reviews. Possui delivery? Não. Grau de recomendação interno do Foodhunter: 0.63.


## 4. Seleção Final e Armazenamento
Agora que o dado está pronto para consumo, vamos descartar colunas intermediárias (como os logs e dados normalizados separadamente) e salvar apenas o que o Banco Vetorial e o Front-end vão precisar.

In [5]:
# Colunas estritamente necessárias para o RAG e pro App
colunas_gold = [
    'business_id', 
    'name', 
    'city', 
    'categories', 
    'price_range', 
    'has_delivery', 
    'stars', 
    'gold_score', 
    'rag_context', 
    'embedding_list' # Crucial para a busca vetorial (Similaridade de Cosseno)
]

df_gold = df[colunas_gold].copy()

# Salvando a base Gold localmente
caminho_saida = os.path.join(diretorio_atual, 'gold_data.parquet')
df_gold.to_parquet(caminho_saida, engine='pyarrow', index=False)

print(f"🏆 Camada Gold concluída! Dataset pronto para o RAG.")
print(f"Salvo em: {caminho_saida}")

# Schema final
df_gold.info()

🏆 Camada Gold concluída! Dataset pronto para o RAG.
Salvo em: c:\Faculdade - IA\Projeto AC2\RAG-Sapato\gold\gold_data.parquet
<class 'pandas.DataFrame'>
RangeIndex: 34987 entries, 0 to 34986
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   business_id     34987 non-null  str    
 1   name            34987 non-null  str    
 2   city            34987 non-null  str    
 3   categories      34987 non-null  str    
 4   price_range     34987 non-null  int64  
 5   has_delivery    34987 non-null  bool   
 6   stars           34987 non-null  float64
 7   gold_score      34987 non-null  float64
 8   rag_context     34987 non-null  str    
 9   embedding_list  34987 non-null  object 
dtypes: bool(1), float64(2), int64(1), object(1), str(5)
memory usage: 15.1+ MB
